# 02 · Multimodal Model Training & Inference Pipeline

> **阿里云 PAI DSW · Qwen2-VL-2B-Instruct · NF4 4-bit · FP16 · QLoRA · Base/LoRA Caption 对比**

基于阿里云教程的 **Step + Markdown + 可执行 Cell + 输出展示** 格式，但本项目使用**单张 Tesla V100 16GB，在 DSW 中完成训练和推理**，不启动原教程的 8 卡 DLC / EAS，因此更适合低成本复现。

本次实际实验：20 张 CAM_FRONT → 19 训练 / 1 验证；LoRA 参数 **2,179,072（0.0985%）**；微调后的输出由 7 项长列表变成 1 句 Caption。**这些只是既往实验数据，不是本 Notebook 已自动复跑后的结果**。

> **运行策略**：缓存和已训练 Adapter 优先复用；训练和覆盖会显式确认。已成功微调过的 DSW 实例可以直接从 Step 2 加载现有 JSONL，然后跳到微调后对比。

## Step 0 · 环境准备 / Restart Kernel

沿用 Notebook 01 的 `.env`（项目根目录）、Python 3.11、PyTorch `2.9.1+cu126`、transformers `4.52.4`、peft `0.15.2`、bitsandbytes `0.48.0`。

首次安装请在 DSW Terminal 中执行：

```bash
cd aliyun-autonomous-driving-multimodal-pipeline
python -m pip install -r requirements-data.txt -c constraints-v100.txt
python -m pip install -r requirements-train.txt -c constraints-v100.txt
python -m pip check
```

**安装依赖后选择 Kernel → Restart Kernel**，再从 Step 1 开始。V100 使用 FP16 + eager attention，不要套用针对 A100 的 BF16 / FlashAttention-2 或强制重装 PyPaimon（会破坏当前 CUDA 组合）。

## Step 1 · 加载 `.env`、检查 GPU / 依赖

In [ ]:
# Step 1：定位仓库根目录；无论 Notebook 从根目录还是 notebooks/ 打开都适用
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / 'aliyun-autonomous-driving-multimodal-pipeline',
              Path.home() / 'workspace' / 'aliyun-autonomous-driving-multimodal-pipeline']
ROOT = next((p for p in candidates if (p / 'config.py').is_file()
             and (p / 'multimodal_data_processing').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('未找到项目根目录。请先 git clone，并在仓库的 notebooks/ 目录打开本教程。')
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
print('Project root:', ROOT)

In [ ]:
# Step 2：读取 .env → Python 配置；不输出任何密钥原文
from config import settings, check_environment

check_environment()
print('DLF Warehouse:', settings.dlf_warehouse)
print('Raw / Enriched:', settings.raw_table, settings.enriched_table)
print('Work dir:', settings.work_dir)
print('DSW 配置检查:', {
    'DLF AK/SK': bool(settings.access_key_id and settings.access_key_secret),
    'Vision': bool(settings.vision_base_url and settings.vision_key and settings.vision_model != 'your-vision-model'),
    'Embedding': bool(settings.embedding_url and settings.embedding_key),
})

## Step 2 · 读取 Paimon 或复用已有 19/1 JSONL 数据

当前 Notebook **优先复用** `/mnt/systemDisk/ad_multimodal/vlm_training/train.jsonl` 与 `validation.jsonl`。这样既保留原来的验证图片，也不会因为重新随机划分而改变基线。

如果不存在 JSONL，再显式允许从增强 Paimon 表导出。**20 张相邻帧高度相似，1 张验证只能用于工程烟雾测试**。

In [ ]:
from multimodal_training_inference.prepare_dataset import (
    load_records, verify_existing_dataset, select_valid_rows,
    split_records, write_dataset, dataset_statistics,
)
from pathlib import Path
from IPython.display import display
import pandas as pd
import json

DATA_DIR = settings.dataset_dir
RUN_EXPORT_FROM_PAIMON = False  # 仅首次无 JSONL 时改 True

if (DATA_DIR / 'train.jsonl').exists() and (DATA_DIR / 'validation.jsonl').exists():
    verify_existing_dataset(DATA_DIR)
elif RUN_EXPORT_FROM_PAIMON:
    from multimodal_data_processing.paimon_io import connect_catalog, read_table
    df = read_table(connect_catalog().get_table(settings.enriched_table))
    good = select_valid_rows(df, dataset_tag=1)
    train_df, val_df = split_records(good, settings.val_ratio, settings.seed)
    print('训练集统计:', dataset_statistics(train_df))
    write_dataset(train_df, val_df, DATA_DIR, settings.seed)
else:
    raise FileNotFoundError('未找到训练 JSONL；请先完成 Notebook 01，或将 RUN_EXPORT_FROM_PAIMON=True')

train_records = load_records(DATA_DIR / 'train.jsonl')
val_records = load_records(DATA_DIR / 'validation.jsonl')
print('Train:', len(train_records), 'Validation:', len(val_records))
sample = val_records[0]
print('验证图片:', sample['fileName'])

## Step 3 · 训练数据预览与统计

先确认图像显示正常、Caption 不为空，再开始模型加载。对训练集统计只描述当前样本，不能代表完整 nuScenes 分布。

In [ ]:
from PIL import Image

print('训练样例:', train_records[0]['fileName'])
print('Caption:', train_records[0]['caption'])
with Image.open(train_records[0]['image']) as im:
    display(im.convert('RGB').resize((680, 382)))

lengths = [len(r['caption']) for r in train_records]
print('Caption 平均字符数:', round(sum(lengths)/len(lengths), 2))
print('Caption 字符数区间:', min(lengths), '～', max(lengths))
print('Validation Caption:', sample['caption'])

In [ ]:
# 可视化 Caption 长度分布；matplotlib 仅用于展示，不影响模型训练
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3))
    plt.hist(lengths, bins=min(8, len(lengths)))
    plt.title('Training caption length distribution')
    plt.xlabel('Characters'); plt.ylabel('Count'); plt.grid(alpha=.25)
    plt.show()
except ImportError:
    print('未安装 matplotlib，已在上一步输出数值统计。')

## Step 4 · ModelScope 模型下载 / 4-bit NF4 加载

使用 `Qwen/Qwen2-VL-2B-Instruct`。`AutoProcessor.from_pretrained()` 读取输入编码规则；`from_pretrained()` 加载预训练模型权重，**不等于重新预训练**。NF4 压缩基础权重，LoRA 使用 FP16 进行计算与更新。

首次运行会通过 ModelScope 下载模型到云盘；已缓存的模型会被复用。加载后 GPU 显存会上升。

In [ ]:
import torch
from multimodal_training_inference.model_loader import (
    require_cuda, resolve_model_dir, load_base_model,
)

require_cuda()
print('V100 使用 float16 + eager attention；不启用 bf16 / flash_attention_2')
MODEL_DIR = resolve_model_dir()
print('MODEL_DIR:', MODEL_DIR)

In [ ]:
# 重复运行本 Cell 会重新占用显存；如果已挂载 LoRA 请勿再次加载
from peft import PeftModel
if 'model' in globals():
    print('当前 Kernel 已有 model:', type(model).__name__)
    print('如需重新加载模型，请先重启 Kernel 或明确释放旧模型。')
else:
    model, processor, MODEL_DIR = load_base_model(MODEL_DIR)
    print('模型加载成功:', type(model).__name__)
    print('显存 GiB:', round(torch.cuda.memory_allocated()/2**30, 2))

## Step 5 · Base Model 基线推理（同一张验证图片）

我们固定 Prompt、图像、`max_new_tokens=128`、`do_sample=False`；保存微调前的 Caption 以供之后比较。已存在的 `baseline_prediction.json` **直接复用**，防止重复运行时覆盖历史结果。

In [ ]:
from multimodal_training_inference.inference import PROMPT, predict_caption

baseline_path = DATA_DIR / 'baseline_prediction.json'
if baseline_path.exists():
    baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
    assert Path(baseline['image']).resolve() == Path(sample['image']).resolve(), '基线与验证图片不一致'
    assert baseline['reference_caption'] == sample['caption'], 'Caption 不一致'
    assert baseline['prompt'] == PROMPT, '保存的 Prompt 与源码不同；需要保持同一 Prompt 才能公平比较'
    print('读取已保存基线:', baseline_path)
else:
    assert not isinstance(model, PeftModel), '基线必须使用未挂载 LoRA 的基础模型'
    predicted = predict_caption(model, processor, sample['image'], PROMPT, max_new_tokens=128)
    baseline = {
        'image_id': sample['id'], 'image': sample['image'],
        'prompt': PROMPT, 'reference_caption': sample['caption'],
        'baseline_caption': predicted, 'max_new_tokens': 128,
        'do_sample': False, 'preprocessing_max_tokens': settings.image_max_tokens,
    }
    baseline_path.write_text(json.dumps(baseline, ensure_ascii=False, indent=2), encoding='utf-8')

print('GPT 参考:', baseline['reference_caption'])
print('Qwen2-VL 微调前:', baseline['baseline_caption'])

## Step 6 · LoRA 原理与挂载

基础模型参数冻结，只在 Attention 投影层（`q_proj/k_proj/v_proj/o_proj`）插入低秩 A/B 矩阵。实验配置 `r=8`, `alpha=16`, `dropout=0.05`；本次可训练参数约 218 万。

**重要：同一个 `model` 只能挂载一次 LoRA**。运行时若已有 Adapter，将直接复用现有训练结果，而不会重复训练。

In [ ]:
from multimodal_training_inference.train_lora import (
    attach_lora_once, encode_training_sample, train,
)

ADAPTER_DIR = settings.lora_output_dir
adapter_exists = (ADAPTER_DIR / 'adapter_config.json').is_file()

if adapter_exists:
    print('已找到训练好的 LoRA Adapter:', ADAPTER_DIR)
    print('跳过新的 LoRA 挂载；Step 10 会加载 Adapter 推理')
elif isinstance(model, PeftModel):
    print('当前模型已经是 PEFT 模型，不重复挂载')
else:
    model = attach_lora_once(model, rank=8)
    print('准备进行一张样本的 Forward/Backward 检查')

## Step 7 · 构造训练标签 `labels=-100`

完整消息为 `Image + User Prompt + Assistant Caption`。模型会自回归预测 Token，但**只对 Assistant Caption 对应的位置计算交叉熵**。用户 Prompt 和图片 Token 都被标为 `-100`，从 Loss 中忽略。

In [ ]:
# 这一单元格会生成一个训练 Tensor，不更新权重
if adapter_exists:
    print('已有训练好的 Adapter；如只想评估可跳过标签检查。')
else:
    batch = encode_training_sample(train_records[0], processor, prompt=PROMPT)
    print('input_ids:', tuple(batch['input_ids'].shape))
    print('labels:', tuple(batch['labels'].shape))
    print('有效监督 Token:', int((batch['labels'] != -100).sum()))
    valid_label_ids = batch['labels'][0][batch['labels'][0] != -100]
    print('实际训练文本:', processor.decode(valid_label_ids, skip_special_tokens=True))
    del batch

## Step 8 · Forward / Loss / Backward 单样本检查

这里仅检查 FP16 + NF4 + LoRA 的计算链路，不调用 `optimizer.step()`，因此不会改变权重。仅当未存在训练好的 Adapter 时执行。

In [ ]:
RUN_GRADIENT_SMOKE_TEST = False  # 首次训练前可改 True

if RUN_GRADIENT_SMOKE_TEST:
    assert not adapter_exists, '已保存 Adapter；不要在已有模型上继续训练测试'
    assert isinstance(model, PeftModel), '请先在 Step 6 挂载 LoRA'
    model.config.use_cache = False
    model.train()
    model.zero_grad(set_to_none=True)
    test_batch = encode_training_sample(train_records[0], processor, PROMPT)
    outputs = model(**test_batch)
    loss = outputs.loss
    assert torch.isfinite(loss), 'Loss 非有限值'
    loss.backward()
    grads = [p.grad for n,p in model.named_parameters()
             if 'lora_' in n and p.requires_grad and p.grad is not None]
    assert grads and all(torch.isfinite(g).all() for g in grads), 'LoRA 梯度无效'
    norm = torch.sqrt(sum(g.detach().float().square().sum() for g in grads))
    assert norm.item() > 0
    print('Loss:', loss.item(), 'LoRA gradient norm:', norm.item())
    model.zero_grad(set_to_none=True)
    del test_batch, outputs, loss, grads
else:
    print('跳过单样本梯度检查')

## Step 9 · 19 张样本 QLoRA 训练并保存 Adapter

实验配置：**1 Epoch、梯度累积 4、学习率 `1e-4`、LoRA rank=8**。实际训练会占用 GPU、耗时且产生费用。已有 Adapter 时禁止覆盖；若需要新实验，应更改输出目录并重新加载基础模型。

In [ ]:
RUN_TRAINING = False  # 首次需要训练时改 True
loss_history = None

if RUN_TRAINING:
    assert not adapter_exists, f'{ADAPTER_DIR} 已有 Adapter，先备份或更换实验目录'
    assert isinstance(model, PeftModel), '请先完成 Step 6 挂载 LoRA'
    loss_history = train(train_records, model, processor,
                         epochs=1, accumulation=4, learning_rate=1e-4,
                         seed=settings.seed)
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(ADAPTER_DIR))
    processor.save_pretrained(str(ADAPTER_DIR))
    (ADAPTER_DIR / 'train_summary.json').write_text(
        json.dumps({'model_id':settings.model_id, 'prompt':PROMPT,
                    'loss_history':loss_history, 'train_samples':len(train_records),
                    'epochs':1, 'accumulation':4, 'learning_rate':1e-4},
                   ensure_ascii=False, indent=2), encoding='utf-8')
    adapter_exists = True
    print('LoRA Adapter 已保存:', ADAPTER_DIR)
else:
    print('训练未执行；已有 Adapter 可直接进入对比')

In [ ]:
# Loss 趋势（训练历史来自当前 Cell 或已保存的 train_summary.json）
if loss_history is None and (ADAPTER_DIR / 'train_summary.json').is_file():
    loss_history = json.loads((ADAPTER_DIR / 'train_summary.json').read_text(encoding='utf-8')).get('loss_history')

if loss_history:
    print('Loss 首值:', round(loss_history[0], 4), '末值:', round(loss_history[-1], 4))
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(8, 3))
        plt.plot(range(1, len(loss_history)+1), loss_history, marker='o')
        plt.xlabel('Training sample step'); plt.ylabel('Cross-entropy Loss')
        plt.title('Qwen2-VL QLoRA Loss'); plt.grid(alpha=.3); plt.show()
    except ImportError:
        print('matplotlib 未安装：已输出 Loss 首末值')
else:
    print('暂无 Loss history。')

## Step 10 · 同图、同 Prompt 的 LoRA 推理对比

只有在 `adapter_config.json` 已成功保存后才执行。使用 **Step 5 保存的同一张验证图** 和完全相同的生成参数。

注意：Prompt 变短、列表变为句子不等价于视觉理解能力更强；20 张相似连续帧的 19/1 验证不足以评估泛化。

In [ ]:
from peft import PeftModel

adapter_exists = (ADAPTER_DIR / 'adapter_config.json').is_file()
assert adapter_exists, '未发现 LoRA Adapter。请先在 Step 9 完成训练。'
assert Path(baseline['image']).resolve() == Path(sample['image']).resolve()
assert baseline['reference_caption'] == sample['caption']
assert baseline.get('max_new_tokens', 128) == 128
assert baseline.get('do_sample', False) is False
assert baseline.get('preprocessing_max_tokens', settings.image_max_tokens) == settings.image_max_tokens

if not isinstance(model, PeftModel):
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR), is_trainable=False)
    print('已从云盘恢复 LoRA Adapter')
else:
    print('使用当前 Kernel 已挂载的 LoRA Adapter')

finetuned_caption = predict_caption(model, processor, sample['image'],
                                     prompt=baseline['prompt'], max_new_tokens=128)
comparison = {**baseline, 'finetuned_caption': finetuned_caption,
              'disclaimer':'19/1 adjacent-frame qualitative comparison only'}
comparison_path = DATA_DIR / 'inference_comparison.json'
comparison_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')

print('1. GPT 参考 Caption:', baseline['reference_caption'])
print('2. Qwen2-VL 微调前:', baseline['baseline_caption'])
print('3. Qwen2-VL 微调后:', finetuned_caption)
print('已保存:', comparison_path)

### 实际运行结果示例（来自之前的 DSW 实验）

**GPT 参考**

> A city street approaches a signalized intersection with marked crosswalks, cars ahead, and pedestrians on both sidewalks. Multi-story brick buildings line the road under a cloudy daytime sky.

**微调前 Qwen2-VL**：输出包含 Street View / Buildings / Traffic / Pedestrians / Street Elements / Weather / Risk 的多项列表，在 128 Token 限制下被截断。

**微调后 Qwen2-VL + LoRA**

> A busy urban street with brick buildings on both sides, traffic lights, pedestrians, and vehicles, with a cloudy sky overhead.

这展示了**输出格式的变化**，但不能凭单张相邻帧证明跨场景泛化提升。下方 Step 10 的代码会根据你当前 DSW 上的 Adapter 重新生成真实结果。


In [ ]:
# 最终可视化：验证图片 + 三份文字对照
with Image.open(sample['image']) as im:
    display(im.convert('RGB').resize((800, 450)))

display(pd.DataFrame([
    {'模型': 'GPT 参考', 'Caption': baseline['reference_caption']},
    {'模型': 'Base Qwen2-VL', 'Caption': baseline['baseline_caption']},
    {'模型': 'Qwen2-VL + LoRA', 'Caption': comparison['finetuned_caption']},
]))

## ✅ Part 2 完成：检查保存的产物

关键产物（均在 DSW 云盘，不上传 GitHub）：

- `vlm_training/train.jsonl`、`validation.jsonl`、`images/`
- `vlm_training/baseline_prediction.json`、`inference_comparison.json`
- `qwen2vl_lora/adapter_config.json` + Adapter 权重

如果需要再次运行：**重启 Kernel → Step 1/2/4 → Step 5 复用基线 → Step 6 复用 Adapter → Step 10 对比**，避免意外二次挂载和重新训练。

[回到 Notebook 01](01_multimodal_data_processing.ipynb) · [查看模型训练源码](../multimodal_training_inference/) · [项目 README](../README.md)

> 说明：Notebook 展示的是可重复执行的工程流程；已知示例 Caption 来自先前的真实实验，而非此次上传代码的实时运行输出。